### CCGP Decoding

In [ ]:
# Cell 1 — Imports and parameters
import os
import pickle
import numpy as np
import pandas as pd
 
import decoding_CCGP
 
PICKLE_DIR = os.path.expanduser('~/Documents/lab/vca2/session_pickles')
OUTPUT_DIR = os.path.expanduser('~/Documents/lab/vca2/ccgp_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
SESSION_LABELS = ['SS_NN', 'SS_NF', 'OS_NN', 'OS_NF', 'OS_NFS']
 
# Match threshold used elsewhere
MIN_NEURONS_ACROSS_ANIMALS = 33
 
# Decoder parameters
N_ITERATIONS_TRUE = 100
N_ITERATIONS_NULL = 20
N_PERMUTATIONS_NULL = 500
C = 1.0
 
SAVE_RESULTS = True

In [ ]:
# Cell 2 — Define CCGP fold specifications
# These are fixed for the 2x2 identity x location structure.
# Fold A trains on left, tests on right; Fold B is the reverse.
 
# Identity CCGP: does identity coding at left cup generalize to right cup?
IDENTITY_FOLD_A_TRAIN = {'partner': ['pl'], 'novel': ['nl']}
IDENTITY_FOLD_A_TEST  = {'partner': ['pr'], 'novel': ['nr']}
IDENTITY_FOLD_B_TRAIN = {'partner': ['pr'], 'novel': ['nr']}
IDENTITY_FOLD_B_TEST  = {'partner': ['pl'], 'novel': ['nl']}
 
# Location CCGP: does location coding for partner generalize to novel (and vice versa)?
LOCATION_FOLD_A_TRAIN = {'right': ['pr'], 'left': ['pl']}
LOCATION_FOLD_A_TEST  = {'right': ['nr'], 'left': ['nl']}
LOCATION_FOLD_B_TRAIN = {'right': ['nr'], 'left': ['nl']}
LOCATION_FOLD_B_TEST  = {'right': ['pr'], 'left': ['pl']}
 
SUBCLASS_PTS_KEYS = ['pr', 'pl', 'nr', 'nl']  # keys in session_data['pseudotrials']
 

In [ ]:
# Cell 3 — Helper: build discontinuity_frames from dropped_frame_info. Mirrors logic in other code
def get_discontinuity_frames(dropped_frame_info):
    disc_frames = []
    for start, end in dropped_frame_info['dropped_ranges']:
        disc_frame = int(end) + 1  # first valid frame after the gap
        disc_frames.append(disc_frame)
    return disc_frames

In [ ]:
# Cell 3b — Per-animal within-animal subsample sizes

##### NOTE - if a new animal is added to the dataset, they will need to be added here ##### - MEK
animal_min_neurons = {
    '6175': 86,
    '7845': 68,
    '7981': 49,
    '8020': 33,
    '8689': 58,
    '8929': 34,
    '8946': 46,
}

In [ ]:
# Cell 4 — Main loop: iterate over sessions, then animals
all_ccgp_results = {}

for session_label in SESSION_LABELS:
    print(f'\n{"="*70}')
    print(f'SESSION: {session_label}')
    print(f'{"="*70}')

    pkl_path = os.path.join(PICKLE_DIR, f'{session_label}.pkl')
    if not os.path.isfile(pkl_path):
        print(f'  Pickle not found: {pkl_path} -- skipping')
        continue

    with open(pkl_path, 'rb') as f:
        session_dict = pickle.load(f)

    session_results = {}

    for session_key, session_data in session_dict.items():

        if not session_data.get('included', False):
            print(f'\n  {session_key}: EXCLUDED ({session_data.get("exclusion_reason", "unknown")})')
            continue

        print(f'\n  {"─"*60}')
        print(f'  Animal: {session_key}')
        print(f'  Cells: {session_data["numcells"]} | FR: {session_data["approx_fr"]:.2f} Hz')

        dff_Z = session_data['dff_Z']
        pseudotrials = session_data['pseudotrials']

        missing = [k for k in SUBCLASS_PTS_KEYS if k not in pseudotrials or len(pseudotrials[k]) == 0]
        if missing:
            print(f'  Skipping {session_key}: missing pseudotrials for {missing}')
            continue

        subclass_pts_dict = {k: pseudotrials[k] for k in SUBCLASS_PTS_KEYS}

        for k in SUBCLASS_PTS_KEYS:
            print(f'    {k}: {len(pseudotrials[k])} pseudotrials')

        discontinuity_frames = get_discontinuity_frames(session_data['dropped_frame_info'])
        print(f'  Discontinuity frames: {discontinuity_frames}')

        animal_results = {}

        # Full population + condition neuron shuffle null 
        print(f'\n  >> Identity CCGP (full population)')
        identity_res = decoding_CCGP.ccgp_decoding_loop(
            comparison_label=f'{session_key} identity CCGP',
            subclass_pts_dict=subclass_pts_dict,
            fold_A_train=IDENTITY_FOLD_A_TRAIN,
            fold_A_test=IDENTITY_FOLD_A_TEST,
            fold_B_train=IDENTITY_FOLD_B_TRAIN,
            fold_B_test=IDENTITY_FOLD_B_TEST,
            neural_data=dff_Z,
            n_iterations_true=N_ITERATIONS_TRUE,
            n_iterations_null=N_ITERATIONS_NULL,
            n_permutations_null=N_PERMUTATIONS_NULL,
            discontinuity_frames=discontinuity_frames,
            C=C,
            null_shift=False,
            null_shift_aware=False,
            null_shuffle=False,
            null_condition_shuffle=True,
        )
        animal_results['identity_full'] = identity_res

        print(f'\n  >> Location CCGP (full population)')
        location_res = decoding_CCGP.ccgp_decoding_loop(
            comparison_label=f'{session_key} location CCGP',
            subclass_pts_dict=subclass_pts_dict,
            fold_A_train=LOCATION_FOLD_A_TRAIN,
            fold_A_test=LOCATION_FOLD_A_TEST,
            fold_B_train=LOCATION_FOLD_B_TRAIN,
            fold_B_test=LOCATION_FOLD_B_TEST,
            neural_data=dff_Z,
            n_iterations_true=N_ITERATIONS_TRUE,
            n_iterations_null=N_ITERATIONS_NULL,
            n_permutations_null=N_PERMUTATIONS_NULL,
            discontinuity_frames=discontinuity_frames,
            C=C,
            null_shift=False,
            null_shift_aware=False,
            null_shuffle=False,
            null_condition_shuffle=True,
        )
        animal_results['location_full'] = location_res

        # Within-animal subsampled (per-animal min across sessions)
        animal_id = session_key.split('_')[0]
        n_within = animal_min_neurons.get(animal_id, None)

        if n_within is not None and session_data['numcells'] >= n_within:
            print(f'\n  >> Identity CCGP (subsampled n={n_within}, within-animal min)')
            identity_within = decoding_CCGP.ccgp_decoding_loop(
                comparison_label=f'{session_key} identity CCGP within-animal',
                subclass_pts_dict=subclass_pts_dict,
                fold_A_train=IDENTITY_FOLD_A_TRAIN,
                fold_A_test=IDENTITY_FOLD_A_TEST,
                fold_B_train=IDENTITY_FOLD_B_TRAIN,
                fold_B_test=IDENTITY_FOLD_B_TEST,
                neural_data=dff_Z,
                n_iterations_true=N_ITERATIONS_TRUE,
                n_iterations_null=N_ITERATIONS_NULL,
                n_permutations_null=N_PERMUTATIONS_NULL,
                discontinuity_frames=discontinuity_frames,
                C=C,
                n_neurons_subsample=n_within,
                null_shift=False,
                null_shift_aware=False,
                null_shuffle=False,
                null_condition_shuffle=False,
            )
            animal_results['identity_within_animal'] = identity_within

            print(f'\n  >> Location CCGP (subsampled n={n_within}, within-animal min)')
            location_within = decoding_CCGP.ccgp_decoding_loop(
                comparison_label=f'{session_key} location CCGP within-animal',
                subclass_pts_dict=subclass_pts_dict,
                fold_A_train=LOCATION_FOLD_A_TRAIN,
                fold_A_test=LOCATION_FOLD_A_TEST,
                fold_B_train=LOCATION_FOLD_B_TRAIN,
                fold_B_test=LOCATION_FOLD_B_TEST,
                neural_data=dff_Z,
                n_iterations_true=N_ITERATIONS_TRUE,
                n_iterations_null=N_ITERATIONS_NULL,
                n_permutations_null=N_PERMUTATIONS_NULL,
                discontinuity_frames=discontinuity_frames,
                C=C,
                n_neurons_subsample=n_within,
                null_shift=False,
                null_shift_aware=False,
                null_shuffle=False,
                null_condition_shuffle=False,
            )
            animal_results['location_within_animal'] = location_within
        else:
            print(f'  Skipping within-animal subsampled CCGP: '
                  f'{session_data["numcells"]} cells < per-animal min {n_within}')

        # Across-animal subsampled (no null)
        print(f'\n  >> Identity CCGP (subsampled n={MIN_NEURONS_ACROSS_ANIMALS}, across-animal)')
        identity_across = decoding_CCGP.ccgp_decoding_loop(
            comparison_label=f'{session_key} identity CCGP across-animals',
            subclass_pts_dict=subclass_pts_dict,
            fold_A_train=IDENTITY_FOLD_A_TRAIN,
            fold_A_test=IDENTITY_FOLD_A_TEST,
            fold_B_train=IDENTITY_FOLD_B_TRAIN,
            fold_B_test=IDENTITY_FOLD_B_TEST,
            neural_data=dff_Z,
            n_iterations_true=N_ITERATIONS_TRUE,
            n_iterations_null=N_ITERATIONS_NULL,
            n_permutations_null=N_PERMUTATIONS_NULL,
            discontinuity_frames=discontinuity_frames,
            C=C,
            n_neurons_subsample=MIN_NEURONS_ACROSS_ANIMALS,
            null_shift=False,
            null_shift_aware=False,
            null_shuffle=False,
            null_condition_shuffle=False,
        )
        animal_results['identity_across_animals'] = identity_across

        print(f'\n  >> Location CCGP (subsampled n={MIN_NEURONS_ACROSS_ANIMALS}, across-animal)')
        location_across = decoding_CCGP.ccgp_decoding_loop(
            comparison_label=f'{session_key} location CCGP across-animals',
            subclass_pts_dict=subclass_pts_dict,
            fold_A_train=LOCATION_FOLD_A_TRAIN,
            fold_A_test=LOCATION_FOLD_A_TEST,
            fold_B_train=LOCATION_FOLD_B_TRAIN,
            fold_B_test=LOCATION_FOLD_B_TEST,
            neural_data=dff_Z,
            n_iterations_true=N_ITERATIONS_TRUE,
            n_iterations_null=N_ITERATIONS_NULL,
            n_permutations_null=N_PERMUTATIONS_NULL,
            discontinuity_frames=discontinuity_frames,
            C=C,
            n_neurons_subsample=MIN_NEURONS_ACROSS_ANIMALS,
            null_shift=False,
            null_shift_aware=False,
            null_shuffle=False,
            null_condition_shuffle=False,
        )
        animal_results['location_across_animals'] = location_across

        session_results[session_key] = animal_results

        if SAVE_RESULTS:
            animal_out_path = os.path.join(OUTPUT_DIR, f'{session_key}_ccgp.pkl')
            with open(animal_out_path, 'wb') as f:
                pickle.dump(animal_results, f)
            print(f'\n  Saved: {animal_out_path}')

    all_ccgp_results[session_label] = session_results

    if SAVE_RESULTS:
        session_out_path = os.path.join(OUTPUT_DIR, f'{session_label}_ccgp_all_animals.pkl')
        with open(session_out_path, 'wb') as f:
            pickle.dump(session_results, f)
        print(f'\nSaved session aggregate: {session_out_path}')

print(f'\n{"="*70}')
print('All CCGP decoding complete.')